In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ["NO_ALBUMENTATIONS_UPDATE"] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import tensorflow as tf
print(tf.__version__)

In [ ]:
import keras
print(keras.__version__)

In [ ]:
import logging
logger = tf.get_logger()
logger.setLevel(logging.ERROR)

In [ ]:
import cv2
import math
import time
import shutil 
import random
import itertools
import numpy as np
import pandas as pd 
from glob import glob
import albumentations as A
from tensorflow.keras import ops
from tensorflow import keras
from functools import partial
import matplotlib.pyplot as plt
from tensorflow.keras import backend as K

In [ ]:
# tf.keras.saving.get_custom_objects().clear()
# tf.keras.backend.clear_session() ## to clear other sessions in the environment
# tf.keras.mixed_precision.set_global_policy(tf.keras.mixed_precision.Policy('mixed_float16'))
tf.config.optimizer.set_jit(False) ## enabling XLA

In [ ]:
tf.config.list_physical_devices('GPU')

# Pretraining

In [ ]:
def convert(seconds):
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60     
    return "%d:%02d:%02d" % (hour, minutes, seconds)

In [ ]:
tempfolder="temp_HAM10000"
isExist = os.path.exists(tempfolder)
if not isExist:
    os.makedirs(tempfolder)

In [ ]:
data_root="Datasets/1.HAM10000/rasm/siamese/fold1/"
output_path="./"+tempfolder+"/"

In [ ]:
train_dir=data_root+"train/"
test_dir=data_root+"test/"

In [ ]:
model_name_1 = output_path+"embedding_network_phase1_weights.keras"
model_name_2 = output_path+"embedding_network_phase2_weights.keras"
model_name_3 = output_path+"siamese_trained_model.keras"

In [ ]:
classes=sorted(os.listdir(train_dir))
print(classes)

In [ ]:
num_classes = len(os.listdir(train_dir))
num_train_images = len(glob(train_dir + '/*/*'))
num_test_images = len(glob(test_dir + '/*/*'))

print("num_classes : ", num_classes)
print("num_train_images : ", num_train_images)
print("num_test_images : ", num_test_images)

In [ ]:
EPOCHS=300
IMG_SIZE=224
BATCH_SIZE=16
BATCH_SIZE_SIAMESE=256
LEARNING_RATE=1e-4
dropout_rate=0.3
BUFFER_SIZE=10000
EARLY_STOPPING_PATIENCE=40
AUTO=tf.data.AUTOTUNE

In [ ]:
BUFFER_SIZE=num_train_images
AUTO=tf.data.AUTOTUNE
TRAIN_STEPS_PER_EPOCH=math.ceil(num_train_images/BATCH_SIZE)
TEST_STEPS_PER_EPOCH=math.ceil(num_test_images/BATCH_SIZE)

In [ ]:
# @tf.keras.utils.register_keras_serializable(package="Custom", name="RandomChance")
class RandomChance(tf.keras.layers.Layer):
    def __init__(self, layer, probability, **kwargs):
        super(RandomChance, self).__init__(**kwargs)
        self.layer = layer
        self.probability = probability

    def call(self, inputs, training=True):
        apply_layer = tf.random.uniform([]) < self.probability
        outputs = tf.cond(pred=tf.logical_and(apply_layer, training),
                          true_fn=lambda: self.layer(inputs), false_fn=lambda: inputs,)
        return outputs

    def get_config(self):
        config = super().get_config()
        config.update({"layer": tf.keras.layers.serialize(self.layer),
                        "probability": self.probability,})
        return config

In [ ]:
random_aug = tf.keras.Sequential([
    RandomChance(tf.keras.layers.RandomFlip(mode='horizontal_and_vertical'), 0.1),
    RandomChance(tf.keras.layers.RandomRotation(0.1, fill_mode='nearest'), 0.1),
    RandomChance(tf.keras.layers.RandomTranslation(0.1, 0.1, fill_mode='nearest'), 0.1),
    RandomChance(tf.keras.layers.RandomZoom(0.1, fill_mode='nearest'), 0.1),
])

In [ ]:
def parse_image(filename, label):
    image = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image, channels=3, dct_method="INTEGER_ACCURATE")    
    image = tf.image.resize(image, (IMG_SIZE,IMG_SIZE))
    return image, label

In [ ]:
file_names_train=glob(train_dir + '/*/*')
random.shuffle(file_names_train)    
lbls_train=[classes.index(name.split('/')[-2]) for name in file_names_train]
lbls_train_one_hot = tf.one_hot(lbls_train, depth=num_classes)
train_dataset=tf.data.Dataset.from_tensor_slices((file_names_train, lbls_train_one_hot))

In [ ]:
train_dataset=(
    train_dataset
    .shuffle(buffer_size=BUFFER_SIZE)
    .map(parse_image, num_parallel_calls=AUTO) 
    .map(lambda x, y: (random_aug(x, training=True), y), num_parallel_calls=AUTO)    
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=AUTO)
)

In [ ]:
file_names_test=glob(test_dir + '/*/*')    
lbls_test=[classes.index(name.split('/')[-2]) for name in file_names_test]
lbls_test_one_hot = tf.one_hot(lbls_test, depth=num_classes)
test_dataset=tf.data.Dataset.from_tensor_slices((file_names_test, lbls_test_one_hot))

In [ ]:
test_dataset=(
    test_dataset    
    .map(parse_image, num_parallel_calls=AUTO)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=AUTO)
)

In [ ]:
tr_lb=np.array(lbls_train)
tr_lb_unique=np.unique(tr_lb)
print((np.unique(tr_lb, return_counts=True)))

In [ ]:
from sklearn.utils import compute_class_weight
#formula: n_samples / (n_classes * np.bincount(y))
classWeight=compute_class_weight(class_weight="balanced",classes=tr_lb_unique,y=tr_lb)
classWeight=np.round(classWeight,4)
classWeight_dict = dict(zip(tr_lb_unique, classWeight))
print("classWeight : ",classWeight_dict)

In [ ]:
base_model=tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG_SIZE,IMG_SIZE,3))
base_model.trainable=False

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom", name="SpatialAttention")
class SpatialAttention(tf.keras.layers.Layer):
    def __init__(self, kernel_size=3, **kwargs):
        super(SpatialAttention, self).__init__(**kwargs)
        self.kernel_size = kernel_size
        self.conv = tf.keras.layers.Conv2D(1, kernel_size, padding='same', activation='sigmoid')

    def call(self, inputs):
        avg_out = tf.keras.backend.mean(inputs, axis=-1, keepdims=True)
        max_out = tf.keras.backend.max(inputs, axis=-1, keepdims=True)
        concat = tf.keras.backend.concatenate([avg_out, max_out], axis=-1)
        attention_weights = self.conv(concat)
        return inputs * attention_weights

    def get_config(self):
        config = super().get_config()
        config.update({"kernel_size": self.kernel_size})
        return config

In [ ]:
x = SpatialAttention()(base_model.output)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(dropout_rate, name="dropout1")(x)
x = tf.keras.layers.Dense(512, activation="silu", name="pred1")(x) 
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(dropout_rate, name="dropout2")(x)
x = tf.keras.layers.Dense(128, activation="silu", name="pred2")(x) 
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(dropout_rate, name="dropout3")(x)
output = tf.keras.layers.Dense(num_classes, activation="softmax", name="pred3")(x) 
embedding_network = tf.keras.Model(base_model.input, output, name="Embedding_EfficientNetB0")

In [ ]:
i=1
j=len(embedding_network.layers)
for x in embedding_network.layers:
    print(i, j, x.name, x.output.shape, x.trainable)
    i=i+1
    j=j-1

In [ ]:
# trainable_count = int(np.sum([tf.keras.backend.count_params(w) for w in embedding_network.trainable_weights]))
# non_trainable_count = int(np.sum([tf.keras.backend.count_params(w) for w in embedding_network.non_trainable_weights]))
# total_count = trainable_count + non_trainable_count

# print('Total params: {:,}'.format(total_count))
# print('Trainable params: {:,}'.format(trainable_count))
# print('Non-trainable params: {:,}'.format(non_trainable_count))

In [ ]:
# from tensorflow.python.profiler.model_analyzer import profile
# from tensorflow.python.profiler.option_builder import ProfileOptionBuilder

# def get_flops(model):
#   forward_pass = tf.function(model.call, input_signature=[tf.TensorSpec(shape=(1,) + model.input_shape[1:])])
#   graph_info = profile(forward_pass.get_concrete_function().graph, options=ProfileOptionBuilder.float_operation())
#   flops = graph_info.total_float_ops
#   return flops

In [ ]:
# flops = get_flops(embedding_network)
# macs = flops / 2

In [ ]:
# print(macs/10 ** 9)

In [ ]:
# print(flops/10 ** 9)

# Phase 1

In [ ]:
regularizer=tf.keras.regularizers.l2(1e-6)
for layer in embedding_network.layers:
    for attr in ['kernel_regularizer']:
        if hasattr(layer, attr):
            setattr(layer, attr, regularizer)

In [ ]:
optimizer_fn = tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, clipnorm=2.0)  
metrics = [tf.keras.metrics.CategoricalAccuracy(name='accuracy')]
loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(gamma=5, label_smoothing=0.05)

embedding_network.compile(optimizer = optimizer_fn, loss = loss_fn, metrics = metrics)

In [ ]:
ModelCheckPoint1=tf.keras.callbacks.ModelCheckpoint(filepath=model_name_1,
    monitor='val_accuracy', verbose=1, save_best_only=True, save_weights_only=False)

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=40, min_lr=1e-5)

In [ ]:
%%time
t1=time.time()
history = embedding_network.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=classWeight_dict,
    callbacks=[ModelCheckPoint1, lr_scheduler] 
)
t2=time.time()
time_taken = t2-t1
print("Time taken : ",convert(time_taken)," hh:mm:ss")

In [ ]:
embedding_network = tf.keras.models.load_model(model_name_1)

In [ ]:
test_loss, test_accuracy_by_evaluate = embedding_network.evaluate(test_dataset)

In [ ]:
test_accuracy_by_evaluate_rounded = round((test_accuracy_by_evaluate)*100, 2)

In [ ]:
print(test_accuracy_by_evaluate_rounded)

# Phase 2

In [ ]:
for layer in embedding_network.layers:
    layer.trainable = True

In [ ]:
i=1
j=len(embedding_network.layers)
for x in embedding_network.layers:
    print(i, j, x.name, x.output.shape, x.trainable)
    i=i+1
    j=j-1

In [ ]:
optimizer_fn = tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, clipnorm=2.0)  
metrics = [tf.keras.metrics.CategoricalAccuracy(name='accuracy')]
loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(gamma=5, label_smoothing=0.05)

embedding_network.compile(optimizer = optimizer_fn, loss = loss_fn, metrics = metrics)

In [ ]:
ModelCheckPoint2=tf.keras.callbacks.ModelCheckpoint(filepath=model_name_2,
    monitor='val_accuracy', verbose=1, save_best_only=True, save_weights_only=False)

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=40, min_lr=1e-5)

In [ ]:
%%time
t1=time.time()
history = embedding_network.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=classWeight_dict,
    callbacks=[ModelCheckPoint2, lr_scheduler] 
)
t2=time.time()
time_taken = t2-t1
print("Time taken : ",convert(time_taken)," hh:mm:ss")

In [ ]:
embedding_network = tf.keras.models.load_model(model_name_2)

In [ ]:
test_loss, test_accuracy_by_evaluate = embedding_network.evaluate(test_dataset)

In [ ]:
test_accuracy_by_evaluate_rounded = round((test_accuracy_by_evaluate)*100, 2)

In [ ]:
print(test_accuracy_by_evaluate_rounded)

# Load Siamese Data

In [ ]:
end = '_HAM10000_Siamese'
destination='Results/siamese/1.HAM10000/'

train_csv_file="fold1_train.csv" 
test_csv_file="fold1_test.csv" 

output_filename_train_genuine = "ham10000_siamese_train_genuine_pairs.csv"
output_filename_train_imposter = "ham10000_siamese_train_imposter_pairs.csv"  

output_filename_test_genuine = "ham10000_siamese_test_genuine_pairs.csv"
output_filename_test_imposter = "ham10000_siamese_test_imposter_pairs.csv"  

data_root="Datasets/1.HAM10000/rasm/siamese/fold1/"
path="Datasets/1.HAM10000/"
output_path="./"+tempfolder+"/"

In [ ]:
train_data=[]

In [ ]:
genuine_train=np.array(pd.read_csv(data_root+output_filename_train_genuine, names=["0","1","2"], header=None))
for i in range(genuine_train.shape[0]):
    image1=genuine_train[i][0]
    image2=genuine_train[i][1]
    label=genuine_train[i][2]    
    train_data.append([image1, image2, label])

In [ ]:
imposter_train=np.array(pd.read_csv(data_root+output_filename_train_imposter, names=["0","1","2"], header=None))
for i in range(imposter_train.shape[0]):
    image1=imposter_train[i][0]
    image2=imposter_train[i][1]
    label=imposter_train[i][2]    
    train_data.append([image1, image2, label])

In [ ]:
test_data=[]

In [ ]:
genuine_test=np.array(pd.read_csv(data_root+output_filename_test_genuine, names=["0","1","2"], header=None))
for i in range(genuine_test.shape[0]):
    image1=genuine_test[i][0]
    image2=genuine_test[i][1]
    label=genuine_test[i][2]    
    test_data.append([image1, image2, label])

In [ ]:
imposter_test=np.array(pd.read_csv(data_root+output_filename_test_imposter, names=["0","1","2"], header=None))
for i in range(imposter_test.shape[0]):
    image1=imposter_test[i][0]
    image2=imposter_test[i][1]
    label=imposter_test[i][2]    
    test_data.append([image1, image2, label])

In [ ]:
train_data=np.array(train_data)
test_data=np.array(test_data)

In [ ]:
np.random.shuffle(train_data)
np.random.shuffle(test_data)

In [ ]:
train_data_pair=train_data[:, :2]
train_data_label=train_data[:, 2]
test_data_pair=test_data[:, :2]
test_data_label=test_data[:, 2]

In [ ]:
train_data_label=np.int16(train_data_label)
test_data_label=np.int16(test_data_label)

In [ ]:
print(train_data.shape, test_data.shape)

In [ ]:
def preprocess_image(file_path, label):
    def load_and_preprocess(file_path):        
        image = tf.io.read_file(path + file_path)
        image = tf.image.decode_jpeg(image, channels=3, dct_method="INTEGER_ACCURATE")
        image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method='area')
        return image

    image_a = load_and_preprocess(file_path[0])
    image_b = load_and_preprocess(file_path[1])
    
    return (image_a, image_b), label

In [ ]:
def create_tf_train_dataset(file_paths, labels, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.shuffle(buffer_size=BUFFER_SIZE)
    dataset = dataset.map(lambda x, y: preprocess_image(x, y), num_parallel_calls=AUTO)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=AUTO)
    return dataset

train_ds=create_tf_train_dataset(train_data_pair, train_data_label, BATCH_SIZE)

In [ ]:
def create_tf_test_dataset(file_paths, labels, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    dataset = dataset.shuffle(buffer_size=BUFFER_SIZE)
    dataset = dataset.map(lambda x, y: preprocess_image(x, y), num_parallel_calls=AUTO)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(buffer_size=AUTO)
    return dataset

test_ds=create_tf_test_dataset(test_data_pair, test_data_label, BATCH_SIZE)

# Train

In [ ]:
embedding_network = tf.keras.models.load_model(model_name_2)

In [ ]:
layer_index = [i for i, layer in enumerate(embedding_network.layers) if layer.name == "block4a_expand_conv"][0]
for layer in embedding_network.layers[:layer_index]:
    layer.trainable = False
    
for layer in embedding_network.layers[layer_index:]:
    layer.trainable = True

In [ ]:
i=1
j=len(embedding_network.layers)
for x in embedding_network.layers:
    print(i, j, x.name, x.output.shape, x.trainable)
    i=i+1
    j=j-1

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom", name="euclidean_distance")
def euclidean_distance(vects):
    x, y = vects
    return tf.expand_dims(tf.norm(x - y, axis=1), axis=-1)  # Ensures (batch_size, 1)

In [ ]:
# Siamese Network Inputs
input_1 = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
input_2 = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Pass inputs through embedding network
tower_1 = embedding_network(input_1)
tower_2 = embedding_network(input_2)

merge_layer = keras.layers.Lambda(euclidean_distance, output_shape=(1,))([tower_1, tower_2])
normal_layer = keras.layers.BatchNormalization()(merge_layer)
output_layer = keras.layers.Dense(1, activation="sigmoid")(normal_layer)
siamese = keras.Model(inputs=[input_1, input_2], outputs=output_layer)

In [ ]:
@tf.keras.utils.register_keras_serializable(package="Custom", name="ContrastiveLoss")
class ContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, margin=0.2, reduction=tf.keras.losses.Reduction.SUM_OVER_BATCH_SIZE, name="ContrastiveLoss"):
        super().__init__(reduction=reduction, name=name)  # Properly pass reduction
        self.margin = margin

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)  # Ensure correct dtype
        square_pred = tf.square(y_pred)
        margin_square = tf.square(tf.nn.relu(self.margin - y_pred))
        return tf.reduce_mean((1 - y_true) * square_pred + y_true * margin_square)

    def get_config(self):
        config = super().get_config()  # Get base config (includes reduction)
        config.update({"margin": self.margin})  # Add margin to config
        return config

In [ ]:
siamese.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, clipnorm=2.0),  
    loss=ContrastiveFocalLoss(), 
    metrics=["accuracy"]
)

In [ ]:
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=30, min_lr=1e-5)

In [ ]:
ModelCheckPoint3=tf.keras.callbacks.ModelCheckpoint(filepath=model_name_3,
    monitor='val_accuracy', verbose=1, save_best_only=True, save_weights_only=False)

In [ ]:
EarlyStop=tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True, min_delta=0.001, verbose=1)

In [ ]:
t1=time.time()
history=siamese.fit(
    train_ds, 
    validation_data=test_ds,  
    batch_size=BATCH_SIZE_SIAMESE,
    epochs=EPOCHS,
    class_weight=classWeight_dict,
    callbacks=[ModelCheckPoint3, lr_scheduler] 
)
t2=time.time()
time_taken = t2-t1
print("Time taken : ", convert(time_taken), " hh:mm:ss")

In [ ]:
Hist = {'hist' : history.history} 
df = pd.DataFrame.from_dict(Hist)  
df.to_csv(output_path+'Hist.csv', index = True, header=True)  

# Save Model

In [ ]:
siamese = tf.keras.models.load_model(model_name_3)

In [ ]:
test_loss, test_accuracy_by_evaluate = siamese.evaluate(test_ds)

In [ ]:
test_accuracy_by_evaluate_rounded = round((test_accuracy_by_evaluate)*100, 2)

In [ ]:
print(test_accuracy_by_evaluate_rounded)

In [ ]:
num_epoch = len(history.history['loss'])
executed_epoch_count = len(history.history['loss'])

In [ ]:
savedfilename="epochs_{}".format(executed_epoch_count)+"_test_acc_{}".format(test_accuracy_by_evaluate_rounded)

In [ ]:
name=output_path+"{}".format(savedfilename)+".keras"
siamese.save(name)

In [ ]:
name_weights=output_path+"{}".format(savedfilename)+".weights.h5"
siamese.save_weights(name_weights)

In [ ]:
figpath=output_path+"{}".format(savedfilename)+".pdf"
fig = plt.figure(figsize=(10,4))
epochs_range = range(len(history.history['loss']))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history['accuracy'], label='Train')
plt.plot(epochs_range, history.history['val_accuracy'], label='Val')
plt.legend(loc='lower right')
plt.title('Accuracy')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['loss'], label='Train')
plt.plot(epochs_range, history.history['val_loss'], label='Val')
plt.legend(loc='upper right')
plt.title('Loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.show()
fig.savefig(figpath, format='pdf', bbox_inches='tight')

# Predictions

In [ ]:
for x in range(10):
    print(test_data_pair[x][0], test_data_pair[x][1], test_data_label[x])

In [ ]:
data_root

In [ ]:
test_pairs=[]
l=sorted(glob(data_root+"set1_aug/*/*"))
for x in l:
    path=x
    lis=sorted(os.listdir(str(x)))
    for y in lis:
        d=x.split('/')[-1]
        f1=x+'/'+d+'_Original.png'
        f2=x+'/'+y
        test_pairs.append([f1,f2])

test_pairs=np.array(test_pairs)
df=pd.DataFrame(test_pairs)
df.to_csv(output_path+'test_pairs.csv', header=None, index=False)

In [ ]:
%%time
pr=[]
pa=np.array(pd.read_csv(output_path+'test_pairs.csv', header=None))
for x in pa:
    im1=cv2.imread(x[0])
    im1=cv2.resize(im1, (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_AREA)
    im1=np.expand_dims(im1, axis=0)
    im2=cv2.imread(x[1])
    im2=cv2.resize(im2, (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_AREA)
    im2=np.expand_dims(im2, axis=0)
    pred=siamese.predict([im1,im2], verbose=0)
    pred=np.round(pred[0][0], 4)
    pr.append([x[0], x[1], pred])

pr=np.array(pr) 
df=pd.DataFrame(pr)
df.to_csv(output_path+'test_predictions.csv', header=None, index=False)

In [ ]:
transform_list = ['AutoContrast', 'BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 
                  'ColorJitter', 'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 
                  'Erosion', 'Frost', 'Gamma', 'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 
                  'HueSaturationValue', 'Invert', 'JPEGCompression', 'MedianBlur', 'MotionBlur', 'OpticalDistortion', 
                  'Original', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 'RandomFog', 
                  'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 
                  'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 
                  'Solarize', 'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur']

In [ ]:
dt=np.array(pd.read_csv(output_path+'test_predictions.csv', header=None))
preds=dt[:,2]
preds_new=np.reshape(preds, (-1,54))

df = pd.DataFrame(preds_new, columns=transform_list)  

df.to_csv(output_path+'similarity_matrix.csv', index=False)

In [ ]:
from pytz import timezone 
from datetime import datetime
cur_time=datetime.now(timezone("Asia/Kolkata")).strftime('%d-%m-%Y-%H-%M-%S')
name = cur_time + end + "_" + savedfilename
os.rename(tempfolder, name)
source = '{}'.format(name)+'/'
dest=shutil.move(source, destination)

In [ ]:
print(dest)